# Preprocessing

Companion notebook for the RDDAC [documentation](https://rddac.readthedocs.io). It is written to stand on its own: every step is annotated so the notebook reads top to bottom.

## Walkthrough

1. Run the reference preprocessing on the small bundle
2. Inspect the self-describing output
3. Compare a raw and a processed oil trace
4. Adjust parameters reproducibly
5. The pointcloud stage (needs the full release + simulations)

**Assumptions**: `pip install 'rddac[preprocessing]'` and the small bundle downloaded (`rddac download --small -y`).


In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path

import rddac

DATA_DIR = Path('../data')
# DATA_DIR = Path('./data')   # uncomment instead when running from the repository root
print(rddac.__version__)


## 1. Run the reference preprocessing

The CLI is the supported entry point. Raw files are never modified; processed files land in `<data>/processed/` together with a generated Croissant manifest of the processed layout.


In [ ]:
!rddac preprocess oil force sheet --data-dir {DATA_DIR} -q
sorted(p.name for p in (DATA_DIR / 'processed').iterdir())[:6]


## 2. Inspect the self-describing output

Every group carries the parameter values used plus the per-file cleaning statistics.


In [ ]:
import h5py

path = sorted((DATA_DIR / 'processed').glob('*.h5'))[0]
with h5py.File(path) as f:
    print('force', f['force/data'].shape, '| oil', f['oil_thickness/data'].shape, '| sheet', f['sheet_thickness/data'].shape)
    print({k: f['oil_thickness'].attrs[k] for k in ('n_nan_removed', 'n_hampel_outliers', 'hampel_k')})


## 3. Raw vs processed oil trace


In [ ]:
import matplotlib.pyplot as plt

exp_id = int(path.stem)
with rddac.open_h5(exp_id, data_dir=DATA_DIR) as raw:
    raw_oil = raw['oil_thickness/data'][:]
with h5py.File(path) as f:
    proc_oil = f['oil_thickness/data'][:]

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(raw_oil[:, 0], raw_oil[:, 1], '.', ms=3, alpha=0.5, label='raw')
ax.plot(proc_oil[:, 0], proc_oil[:, 1], lw=1.5, label='processed')
ax.set_xlabel('sensor position / mm'); ax.set_ylabel('oil film / g m$^{-2}$'); ax.legend();


## 4. Adjust parameters reproducibly

`--dump-config` prints the complete defaults as an editable TOML; `--config` applies your variant. Publishing the TOML next to your code makes the variant exactly reproducible.


In [ ]:
!rddac preprocess --dump-config | head -12


## 5. The pointcloud stage

With the full release and the DDACS simulations present (`rddac download`, without `--no-sim`), `rddac preprocess pointcloud` cleans and aligns the scans; on first use it retrains the fin classifier from the labels bundled with the package (one-time, ~30–90 min). The cell below only runs when that data is available.


In [ ]:
sim_dir = DATA_DIR / 'simulation'
if sim_dir.is_dir() and any(sim_dir.iterdir()):
    print('simulations found — run: rddac preprocess pointcloud --workers 8')
else:
    print('simulations not downloaded — skipping the pointcloud stage (see docs).')


## Where to go next

- [Preprocessing documentation](https://rddac.readthedocs.io/en/latest/preprocessing/) — schema, parameters, custom processing
- Notebook 6 — streaming and numpy export over the processed layer
